## This code will implement the Seq2seq model

In [114]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
import pickle
import datasets
import numpy as np
import pandas as pd
import time

## Pipeline creation

In [115]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [116]:
df = pd.read_parquet("/kaggle/input/seq2seq/dataset_seq2seq.parquet")

with open("/kaggle/input/seq2seq/Mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

new_text = []
for i in (df["text"]) :
    new_text.append([mapping[char] for char in i])
df["text"] = new_text

new_context = []
for i in (df["context"]) :
    new_context.append([mapping[char] for char in i])
df["context"] = new_context

In [117]:
new_context_two = []
for i in (df["context_two"]) :
    inter_contest_two = []
    for j in i :
        if "Previous" in j or "<START>" in j :
            inter_contest_two.append(mapping[j])
        else : 
            inter_contest_two.extend([mapping[char] for char in j])
    new_context_two.append(inter_contest_two)
df["context_two"] = new_context_two

In [118]:
mapping_inverse = {i: ch for ch, i in mapping.items()}

In [119]:
for j in new_context_two[0] :
    print(mapping_inverse[j])

Previous :
<START>
Previous :
<START>


In [120]:
df.head()

,context,context_two,text
0,"[114, 190, 128]","[215, 214, 215, 214]","[109, 72, 31, 36, 13, 95, 93, 107, 56, 36, 95,..."
1,"[115, 190, 139]","[215, 214, 215, 214]","[61, 86, 95, 32, 86, 93, 39, 65, 31, 56, 31, 1..."
2,"[115, 179, 139]","[215, 61, 86, 95, 32, 86, 93, 39, 65, 31, 56, ...","[61, 31, 95, 1, 65, 31, 96, 95, 86, 56, 95, 1,..."
3,"[115, 181, 139]","[215, 61, 31, 95, 1, 65, 31, 96, 95, 86, 56, 9...","[64, 107, 93, 95, 108, 41, 31, 31, 36, 13, 29,..."
4,"[115, 208, 139]","[215, 64, 107, 93, 95, 108, 41, 31, 31, 36, 13...","[92, 31, 36, 36, 31, 93, 36, 95, 39, 31, 95, 2..."


In [121]:
df["context"] = df["context"]+df["context_two"]

In [122]:
dataset = datasets.Dataset.from_pandas(df[["context","text"]])
dataset.set_format(type="torch", columns=["context","text"])

split = dataset.train_test_split(test_size=0.2,train_size=0.8, seed=42)

train = split["train"]
test = split["test"]

In [123]:
class SeqDataset(Dataset) :
    def __init__(self, data) :
        self.data = data
        self.length = len(data)

    def __len__(self) :
        return self.length

    def __getitem__(self, i) :
        context = self.data["context"][i]
        text = self.data["text"][i]
        return context,text

In [124]:
train_data = SeqDataset(train)
test_data = SeqDataset(test)

In [125]:
def collate_fn(batch) :
    
    contexts, texts = zip(*batch)  # each already a tensor

    length_context = torch.tensor([len(c) for c in contexts], dtype=torch.long)
    length_text = torch.tensor([len(t) for t in texts], dtype=torch.long)

    
    contexts_padded = pad_sequence(contexts, batch_first=True, padding_value=0)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    
    texts_X = texts_padded[:,:-1]
    texts_Y = texts_padded[:,1:]
        
    return (contexts_padded, texts_X, texts_Y), (length_context, length_text)

In [126]:
 batch_size = 64

if device == "cpu" :

    train_ld = DataLoader(train_data, batch_size=batch_size, shuffle=False, drop_last=True, collate_fn=collate_fn) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_ld = DataLoader(test_data, batch_size=batch_size, shuffle=False, drop_last=True, collate_fn=collate_fn)

else : 
    train_ld = DataLoader(train_data, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                        num_workers=4, shuffle=True, drop_last=True, persistent_workers=True, collate_fn=collate_fn) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_ld = DataLoader(test_data, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                       num_workers=2, shuffle=False, drop_last=True, persistent_workers=True, collate_fn=collate_fn)

List of operations :

- First the entire batch pass through the Encodeur, we extract the h,c from this

- Then we iterate over all the length of the sequence, with the previous input and we use at first the h,c from the Encodeur

- Then we calculate the loss

## Create Seq2seq

In [127]:
class Encodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(Encodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(input_size = embedding_dim, 
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            dropout = 0.25,
                            batch_first=True)

    def forward(self, x, length_batch):
        #Encodeur
        emb_x = self.embed(x)
        #Pack before LSTM to avoid unecessary compute
        packed_x = pack_padded_sequence(emb_x, length_batch, batch_first=True, enforce_sorted=False)                      
        packed_out, (h,c) = self.lstm(packed_x)             
        #Unpack after
        _, _ = pad_packed_sequence(packed_out, batch_first=True) #First right now only h,c
        return (h, c)

In [128]:
class Decodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, masked_mapping, num_layers=1):
        super(Decodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(input_size = embedding_dim, 
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            dropout = 0.25,
                            batch_first=True)
        
        self.final = nn.Linear(hidden_size,vocab_size)
        self.mask = masked_mapping

    def forward(self, x, h, c, length_batch):
        emb_x = self.embed(x)
        #Pack before LSTM to avoid unecessary compute
        packed_x = pack_padded_sequence(emb_x, length_batch, batch_first=True, enforce_sorted=False)                      
        x_lstm, (h,c) = self.lstm(packed_x, (h,c))             
        #Unpack 
        unpacked_out,_ = pad_packed_sequence(x_lstm, batch_first=True) 
        logits = self.final(unpacked_out)
        masked_logits = logits.masked_fill(self.mask, float("-inf"))
  
        return masked_logits, (h, c)

In [129]:
vocab_size = len(mapping)
embedding_size = 64
hidden_size = 256
num_epoch = 250

#Some of the mapping are only for the encodeur so the decodeur can't produce them, we need to mask them from the loss
mapping_inverse = {i: ch for ch, i in mapping.items()}
masked_mapping = list(mapping_inverse.keys())[110:]

mask = torch.zeros(vocab_size, dtype=torch.bool, device=device)
mask[masked_mapping] = True

nb_step_test = len(test_ld)
nb_step_train = len(train_ld)

enco = Encodeur(vocab_size, embedding_size, hidden_size, num_layers=2).to(device)
deco = Decodeur(vocab_size, embedding_size, hidden_size, mask,num_layers=2).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=0) #Just ignore the padding

params = list(enco.parameters()) + list(deco.parameters())
opti = torch.optim.AdamW(params, lr=0.003, weight_decay=1e-3)

sched_warm = torch.optim.lr_scheduler.LinearLR(opti, start_factor=0.2, end_factor=1.0, total_iters=nb_step_train*2)
sched_post = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opti, T_0=nb_step_train*10, T_mult=2, eta_min=0.0001) #1 epoch => 2 => 4 => 8

## Creation boucle entraînement Decodeur

In [ ]:
l_tot = []

#Early stopping
#early_stopping_count = 0
#patience = 5

best_val = float("inf")

scaler = torch.amp.GradScaler()

for epoch in range(num_epoch) :
    
    enco.train()
    deco.train()

    #Create loss per epoch
    l_train = 0.0
    l_test = 0.0
    
    for (context,X, Y), (length_cont,length_text) in iter(train_ld) :
        X = X.to(device, non_blocking = True)
        Y = Y.to(device, non_blocking = True)
        context = context.to(device, non_blocking = True)
        length_text = (length_text-1)

        opti.zero_grad(set_to_none=True)

        #Encodeur part, sees the whole context of the batch and output the hidden and cell state
        h_enco,c_enco = enco(context.to(device),length_cont)
        
        h_dec = h_enco
        c_dec = c_enco
        
        with torch.amp.autocast(device_type="cuda"):
            logits, (h_dec, c_dec) = deco(X,h_dec,c_dec,length_text)
            loss = loss_fn(logits.reshape(-1,vocab_size),Y.reshape(-1))

        scaler.scale(loss).backward()
        #Gradient clipping to avoid exploding gradient
        torch.nn.utils.clip_grad_norm_(list(enco.parameters()) + list(deco.parameters()), 0.25)

        #Scaler
        scaler.step(opti); scaler.update()
        l_train += loss.item()

        #Scheduler part
        #Warm start
        
        if sched_post.T_cur == 0 and epoch > 10:  #After warm restart decrease the max learning rate
            sched_post.base_lrs[0] = sched_post.base_lrs[0] * 0.7
            sched_post.eta_min = sched_post.eta_min * 1.25
            print(f"Decrease {sched_post.base_lrs[0]}, {sched_post.eta_min}")

        step_scheduler = sched_warm if epoch > 10 else sched_post
        step_scheduler.step()

    #Test data part
    
    enco.eval()
    deco.eval()
    
    with torch.inference_mode():
        with torch.amp.autocast("cuda"):
            for (context, X, Y), (length_cont,length_text) in iter(test_ld) :
                X = X.to(device, non_blocking = True)
                Y = Y.to(device, non_blocking = True)
                context = context.to(device, non_blocking = True)
                length_text = length_text-1

                #Encodeur part, sees the whole context of the batch and output the hidden and cell state
                h_enco,c_enco = enco(context.to(device),length_cont)
        
                h_dec = h_enco
                c_dec = c_enco

                logits, (h_dec, c_dec) = deco(X,h_dec,c_dec,length_text)
                masked_logits = logits.masked_fill(mask, float("-inf"))
                loss = loss_fn(masked_logits.reshape(-1,vocab_size),Y.reshape(-1))
                l_test += loss.item()

        print(epoch,np.exp(l_test/nb_step_test),"\n")
     
        #Record the loss of the epoch
        l_tot.append(l_test); 

        if l_test < best_val :
            best_val = l_test
#            early_stopping_count = 0
            torch.save({
                "epoch": epoch,
                "encoder_state_dict": enco.state_dict(),
                "decoder_state_dict": deco.state_dict(),
                "optimizer_state_dict": opti.state_dict(),
                "scheduler_state_dict": sched_post.state_dict(),
                "val_loss": l_test,
            }, "model1")
        
#        elif l_test >= best_val :
#            early_stopping_count += 1

#        if early_stopping_count == patience :
#            print("Early Stopping")
#            break 

print(f"Liste of offset used : {list_offset}")